# QPE: estimating an eigenphase

PHYS 3810, session 2. First complete `02_qft.ipynb` (20 minutes). This lesson supplies the remaining **30-minute core**, with a **25-minute expansion** for a 75-minute combined session. These are proposed timings; classroom pacing still needs rehearsal.

**Before class:** run all cells to verify the environment. **In class:** restart and clear outputs, predict each result, then execute the relevant cell. Preclass/reference checks and expansion cells still execute during required validation.

We estimate $\phi$ in $U|u\rangle=e^{2\pi i\phi}|u\rangle$, modulo one. Here we deliberately construct simple unitaries with known eigenstates and phases so we can verify the algorithm. This is a teaching experiment, not a benchmark or a free way to prepare unknown eigenstates.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from importlib.metadata import version

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "phys3810" / "qpe.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Keep the extracted course folder together and launch Jupyter from its root.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 20-30 minutes: kickback, registers, and controlled powers

The canonical source accepts **1-6 counting qubits**, and a numeric gate/circuit on **1-2 target qubits**. It returns an unmeasured block; **the caller prepares the target before composing it**. Counting qubit `j` controls $U^{2^j}$. The target follows the counting register in the circuit.

For an eigenstate input, controlled powers produce the counting amplitudes $e^{2\pi i x\phi}/\sqrt{N}$, with $N=2^m$. Our positive-sign QFT convention means the final step is its **inverse with swaps included**. The final counting amplitude is

$$a_y=\frac1N\sum_{x=0}^{N-1} e^{2\pi i x(\phi-y/N)}.$$

Measure only `count[j]` into classical bit `j`. The rightmost displayed bit is counting qubit 0; decode with `int(bitstring, 2)/N`, with **no reversal**. Full statevectors have tensor order $|\mathrm{target}\rangle|\mathrm{count}\rangle$.

The helpers below prepare a known target and read out a copy. They import the algorithm; they do not maintain another QPE implementation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import ClassicalRegister, QuantumCircuit
from qiskit.circuit.library import PhaseGate, phase_estimation
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Operator, Statevector
from src.phys3810.qpe import qpe_circuit

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert version("qiskit") == "2.6.1", "Select the pinned classroom kernel."
print("Python:", sys.version.split()[0], "| Qiskit:", version("qiskit"))
print("Interpreter:", sys.executable)
ATOL = 2e-11


def phase_amplitudes(phi, width):
    """Independent Fourier-sum expectation; no circuit or SDK reference."""
    size = 2**width
    delta = phi - np.arange(size) / size
    return np.exp(2j * np.pi * np.outer(delta, np.arange(size))).mean(axis=1)


def eigenstate_example(width, phi):
    """P(2*pi*phi) has |1> as a known eigenstate; prepare it explicitly."""
    unitary = PhaseGate(2 * np.pi * phi)
    block = qpe_circuit(width, unitary)
    prepared = QuantumCircuit(*block.qregs)
    prepared.x(block.qregs[1][0])
    prepared.compose(block, inplace=True)
    return unitary, block, prepared


def counting_readout(circuit, width):
    sampled = circuit.copy()
    phase_register = ClassicalRegister(width, "phase")
    sampled.add_register(phase_register)
    sampled.measure(range(width), phase_register)
    return sampled

In [ ]:
# Editable exact example: use a phase on the m-bit grid for this section.
num_counting_qubits = 3
exact_phase = 3 / 8
unitary, qpe_block, exact_circuit = eigenstate_example(num_counting_qubits, exact_phase)
size = 2**num_counting_qubits
assert 0 <= exact_phase < 1 and np.isclose(exact_phase * size, round(exact_phase * size), rtol=0, atol=1e-12),     "Choose an exact grid phase here; use the nonexact section for other phases."
expected_index = int(round(exact_phase * size))
exact_label = format(expected_index, f"0{num_counting_qubits}b")
expected_exact = np.zeros(size)
expected_exact[expected_index] = 1
exact_state = Statevector.from_instruction(exact_circuit)
np.testing.assert_allclose(exact_state.probabilities(list(range(num_counting_qubits))), expected_exact, rtol=0, atol=ATOL)
np.testing.assert_allclose(exact_state.data, np.kron([0, 1], expected_exact), rtol=0, atol=ATOL)
assert exact_circuit.num_clbits == 0
print(f"PASSED: phi={exact_phase:g} gives {exact_label}, decoded as {expected_index}/{size}")
print("Register roles:", [(register.name, len(register)) for register in qpe_block.qregs])
figure = exact_circuit.draw("mpl", fold=30)
display(figure)
plt.close(figure)

## 30-45 minutes: predict exact outcomes, then sample

Before running the next cell, predict the result for $\phi=3/8$ with three counting bits. Then change the parameter cell to $1/8$ and $6/8$, rerun it, and predict again. These labels are deliberately asymmetric: reversing them changes the answer. Keep the target preparation at $|1\rangle$.

For an exact grid phase, interference leaves a single counting label. What would happen with the same phase but fewer counting bits? Why must the target already be an eigenstate? A non-eigenstate can produce several phases; the routine does not check or supply eigenstate preparation.

In [ ]:
shots = 2048
exact_counts = StatevectorSampler(seed=2026).run(
    [counting_readout(exact_circuit, num_counting_qubits)], shots=shots
).result()[0].data.phase.get_counts()
assert exact_counts == {exact_label: shots}
assert exact_circuit.num_clbits == 0
observed_exact = np.array([exact_counts.get(format(y, f"0{num_counting_qubits}b"), 0) / shots for y in range(size)])
figure, axes = plt.subplots(figsize=(8, 3.5))
axes.bar(np.arange(size), observed_exact, label=f"{shots} local samples", alpha=0.7)
axes.plot(np.arange(size), expected_exact, "ko", label="Exact prediction")
ticks = np.unique(np.r_[np.arange(0, size, max(1, size // 16)), expected_index])
axes.set_xticks(ticks, [format(y, f"0{num_counting_qubits}b") for y in ticks], rotation=45)
axes.set(xlabel="Counting bitstring", ylabel="Probability", ylim=(0, 1.15),
         title=f"phi={exact_phase:g}: {exact_label} decodes to {expected_index}/{size}")
axes.legend()
figure.tight_layout()
display(figure)
plt.close(figure)
print("Counts:", exact_counts)
print(f"PASSED: measuring only count gives {exact_label}; int(bits, 2)/{size} = {exact_phase:g}")

## 45-50 minutes: interpret the register

Explain which register carries the estimate and which bit is least significant. The result `011` means $3/8$, not $6/8$. The number of bits sets grid spacing $2^{-m}$; the result represents a phase modulo one.

**Preclass/instructor reference check, not extra class time:** Qiskit 2.6.1's `phase_estimation()` ends with a counting-bit reversal after its inverse `QFTGate`. Below we test that relationship and explicitly undo it for comparison. Our circuit's decoding rule above does not change. The mathematical tests remain independent of this SDK comparison.

In [ ]:
reference = phase_estimation(num_counting_qubits, unitary)
reversed_rows = [int(format(y, f"0{num_counting_qubits}b")[::-1], 2) for y in range(size)]
reversal = np.kron(np.eye(2), np.eye(size)[reversed_rows])
np.testing.assert_allclose(Operator(reference).data, reversal @ Operator(qpe_block).data, rtol=0, atol=ATOL)
for low in range(num_counting_qubits // 2):
    reference.swap(low, num_counting_qubits - low - 1)
np.testing.assert_allclose(Operator(reference).data, Operator(qpe_block).data, rtol=0, atol=ATOL)
print(f"Pinned SDK raw counting convention: {exact_label[::-1]}; our declared convention: {exact_label}")
print("PASSED: complete SDK operator agrees after the explicit output reversal")

## 50-65 minutes: a nonexact phase has a distribution

A finite register cannot encode $1/3$ exactly. Predict where the largest probabilities should lie with three counting bits. Then compare with five bits. The same Fourier sum gives a distribution across all bins; a single measurement need not be the nearest estimate.

Change `nonexact_phase` or the two `precision_bits` below and rerun this cell and the precision report. Labels and expectations follow those parameters. The displayed bars are actual V2 samples; markers are the independently calculated probabilities. Assertions also check exact statevector probabilities and the unchanged target.

In [ ]:
# Editable expansion parameters: two different widths, each within 1..6.
nonexact_phase = 1 / 3
precision_bits = (3, 5)
assert 0 <= nonexact_phase < 1, "Use a phase in [0, 1) for this display."
assert len(precision_bits) == 2 and len(set(precision_bits)) == 2, "Choose two distinct counting widths."
nonexact_probabilities = {}
nonexact_counts = {}
figure, panels = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for panel, width in zip(panels, precision_bits):
    _, _, circuit = eigenstate_example(width, nonexact_phase)
    n = 2**width
    exact = Statevector.from_instruction(circuit)
    probabilities = exact.probabilities(list(range(width)))
    theory = np.abs(phase_amplitudes(nonexact_phase, width))**2
    np.testing.assert_allclose(probabilities, theory, rtol=0, atol=ATOL)
    np.testing.assert_allclose(exact.data, np.kron([0, 1], phase_amplitudes(nonexact_phase, width)), rtol=0, atol=ATOL)
    counts = StatevectorSampler(seed=2026).run(
        [counting_readout(circuit, width)], shots=2048
    ).result()[0].data.phase.get_counts()
    frequencies = np.array([counts.get(format(y, f"0{width}b"), 0) / 2048 for y in range(n)])
    assert sum(counts.values()) == 2048
    tolerance = 6 * np.sqrt(np.maximum(0, theory * (1 - theory)) / 2048) + 1 / 2048
    assert np.all(np.abs(frequencies - theory) <= tolerance)
    nonexact_probabilities[width] = probabilities
    nonexact_counts[width] = counts
    grid = np.arange(n) / n
    panel.bar(grid, frequencies, width=0.8 / n, alpha=0.65, label="2048 local samples")
    panel.plot(grid, theory, "ko", markersize=4, label="Independent probability")
    panel.axvline(nonexact_phase, color="tab:red", linestyle="--", label=f"Input phase {nonexact_phase:.4f}")
    panel.set(ylabel="Probability", title=f"m={width}, N={n}, grid spacing={1/n:g}", ylim=(0, 1.05), xlim=(-0.03, 1.03))
    panel.legend(loc="upper right", fontsize=8)
    print(f"PASSED: m={width}, phi={nonexact_phase:g}, exact distribution and finite samples")
panels[-1].set_xlabel("Decoded estimate y / N (phase modulo one)")
figure.tight_layout()
display(figure)
plt.close(figure)

## 65-75 minutes: precision, probability, and cost

Compare the most likely estimates below. The error uses circular distance because phases are modulo one. Increasing the width refines the grid; it does **not** promise every sampled outcome is closer. Try a phase near one to see probability wrap around to zero.

Each circuit has $m$ controlled-power blocks. If implemented by repeated controlled-U access, they require $1+2+\cdots+2^{m-1}=2^m-1$ uses of U. The source instead classically forms dense 2x2 or 4x4 powers for these bounded demonstrations. State preparation, synthesis, and controlled-U access all have costs. This is not a general quantum-advantage claim.

In [ ]:
for width in precision_bits:
    probabilities = nonexact_probabilities[width]
    n = 2**width
    peak = int(np.argmax(probabilities))
    estimate = peak / n
    error = abs((estimate - nonexact_phase + 0.5) % 1 - 0.5)
    assert error <= 0.5 / n + ATOL
    label = format(peak, f"0{width}b")
    print(f"m={width}: most likely {label} -> {estimate:.6f}; circular error {error:.6f}; peak probability {probabilities[peak]:.6f}; repeated-U cost {n-1}")
print("PASSED: most likely bins are nearest grid estimates; individual samples can be farther away")

## Input-contract check for preclass validation

QPE accepts numeric unitary gates/circuits; it cannot coherently control a measurement or reset. Intentional invalid-input examples are handled inside successful cells, never by an allowed-error or skip-execution tag. Target preparation remains the caller's responsibility.

In [ ]:
measured_unitary = QuantumCircuit(1, 1)
measured_unitary.measure(0, 0)
try:
    qpe_circuit(3, measured_unitary)
except ValueError as error:
    print("PASSED: a measured circuit is rejected:", error)
else:
    raise AssertionError("A measurement was accepted as a unitary input.")

## What this lesson establishes

You have used controlled powers and the inverse QFT to estimate known eigenphases, decoded asymmetric bitstrings, and checked a nonexact outcome distribution. The full test suite additionally checks complex one- and two-qubit eigenstates, superpositions, global phase, and every supported counting width. A global phase of U becomes a relative phase under control and must not be dropped.

For later applications, an eigenphase is not automatically a Hamiltonian eigenvalue: evolution sign, time/scale, wrapping, precision, and the prepared state must be specified. HHL and Shor are not implemented in this milestone. The next independent review checks the combined QFT/QPE conventions before those modules.

References: [IBM phase-estimation API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.phase_estimation), [phase-estimation procedure](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/phase-estimation-procedure), and [bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering).